# Chapter 2 — Observer Pattern
## *Keeping Your Objects in the Know*

**Intent:** Define a one-to-many dependency so that when one object (the **Subject**) changes state, all its dependents (**Observers**) are notified and updated automatically.

### OO Design Principles
- **Strive for loosely coupled designs** between objects that interact.  
  Subject only knows observers implement a single `update()` method — nothing else.

### Book context
Weather station: a `WeatherData` object holds temperature/humidity/pressure.  
Display boards (`CurrentConditions`, `Statistics`, `Forecast`) all need to update when readings change.

In [1]:
from abc import ABC, abstractmethod

# ── Interfaces ───────────────────────────────────────────────────────────────
class Observer(ABC):
    @abstractmethod
    def update(self, temp: float, humidity: float, pressure: float): ...

class Subject(ABC):
    @abstractmethod
    def register_observer(self, o: Observer): ...
    @abstractmethod
    def remove_observer(self, o: Observer): ...
    @abstractmethod
    def notify_observers(self): ...

In [2]:
# ── Concrete Subject ─────────────────────────────────────────────────────────
class WeatherData(Subject):
    def __init__(self):
        self._observers: list[Observer] = []
        self._temperature = self._humidity = self._pressure = 0.0

    def register_observer(self, o: Observer):
        self._observers.append(o)

    def remove_observer(self, o: Observer):
        self._observers.remove(o)

    def notify_observers(self):
        for o in self._observers:
            o.update(self._temperature, self._humidity, self._pressure)

    def set_measurements(self, temp: float, humidity: float, pressure: float):
        self._temperature = temp
        self._humidity = humidity
        self._pressure = pressure
        self.notify_observers()

In [3]:
# ── Concrete Observers ───────────────────────────────────────────────────────
class CurrentConditionsDisplay(Observer):
    def update(self, temp, humidity, pressure):
        print(f"[Current] {temp}°F, {humidity}% humidity")


class StatisticsDisplay(Observer):
    def __init__(self):
        self._temps: list[float] = []

    def update(self, temp, humidity, pressure):
        self._temps.append(temp)
        print(f"[Stats] avg={sum(self._temps)/len(self._temps):.1f}, "
              f"max={max(self._temps)}, min={min(self._temps)}")


class ForecastDisplay(Observer):
    def __init__(self):
        self._last = 0.0

    def update(self, temp, humidity, pressure):
        if pressure > self._last:
            print("[Forecast] Improving weather on the way!")
        elif pressure < self._last:
            print("[Forecast] Watch out for cooler, rainy weather.")
        else:
            print("[Forecast] More of the same.")
        self._last = pressure

In [4]:
# ── Demo ─────────────────────────────────────────────────────────────────────
weather = WeatherData()
current = CurrentConditionsDisplay()
stats   = StatisticsDisplay()
forecast = ForecastDisplay()

weather.register_observer(current)
weather.register_observer(stats)
weather.register_observer(forecast)

print("--- Measurement 1 ---")
weather.set_measurements(80, 65, 30.4)
print("--- Measurement 2 ---")
weather.set_measurements(82, 70, 29.2)

print("\n--- Remove forecast observer ---")
weather.remove_observer(forecast)
weather.set_measurements(78, 90, 29.2)

--- Measurement 1 ---
[Current] 80°F, 65% humidity
[Stats] avg=80.0, max=80, min=80
[Forecast] Improving weather on the way!
--- Measurement 2 ---
[Current] 82°F, 70% humidity
[Stats] avg=81.0, max=82, min=80
[Forecast] Watch out for cooler, rainy weather.

--- Remove forecast observer ---
[Current] 78°F, 90% humidity
[Stats] avg=80.0, max=82, min=78


## Python built-in: using `__setattr__` and weakref observers

In [7]:
from typing import Callable

_listeners: dict[str, list[Callable]] = {}

_listeners.setdefault('data', []).append(lambda t, h, p: print(f"  Listener A → temp={t}"))
_listeners.setdefault('data', [])

_listeners



{'data': [<function __main__.<lambda>(t, h, p)>]}

In [ ]:
# Pythonic version using a simple event system


class EventEmitter:
    def __init__(self):
        self._listeners: dict[str, list[Callable]] = {}

    def on(self, event: str, callback: Callable):
        self._listeners.setdefault(event, []).append(callback)

    def emit(self, event: str, *args, **kwargs):
        for cb in self._listeners.get(event, []):
            cb(*args, **kwargs)


emitter = EventEmitter()
emitter.on("data", lambda t, h, p: print(f"  Listener A → temp={t}"))
emitter.on("data", lambda t, h, p: print(f"  Listener B → pressure={p}"))

emitter.emit("data", 75, 60, 30.1)

  Listener A → temp=75
  Listener B → pressure=30.1


## Interview cheat-sheet

| Question | Answer |
|---|---|
| Push vs Pull model? | Push: subject sends data in `update()`. Pull: observer calls getters on subject. Book recommends pull for flexibility. |
| Python stdlib equivalent? | `tkinter` variables, `asyncio` events, Django signals. |
| Pitfall? | Memory leaks if observers aren't removed; use `weakref` in long-lived apps. |
| Difference from Pub/Sub? | Observer has a direct reference to subject; Pub/Sub adds a message broker in between. |

**Pattern family:** Behavioral